In [ ]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 57.2 MB/s eta 0:00:00


# **SETUP**

In [ ]:

from google.colab import drive
import os
import sys
from pathlib import Path
import yaml
import torch
import importlib
import gc  # Garbage Collector per  RAM
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import WandbLogger

print(" STEP 5: Fine-tuning COCO model on Cityscapes")
print("=" * 90)


print("Mounting Google Drive and setting up paths...")
drive.mount('/content/drive', force_remount=True)

project_folder = "/content/drive/MyDrive/FAIMDL Project"
repo_name = "MaskArchitectureAnomaly_CourseProject"
repo_path = os.path.join(project_folder, repo_name)
eomt_folder = os.path.join(repo_path, "eomt")
data_path = os.path.join(project_folder, "Cityscapes_Dataset")

if not os.path.exists(project_folder):
    print(f" ERRORE: the folder does not exist {project_folder}")
    sys.exit(1)

if not os.path.exists(repo_path):
    print(f" Repo does not exist")
    os.chdir(project_folder)
    os.system("git clone https://github.com/AndreaPassera/MaskArchitectureAnomaly_CourseProject.git")

os.chdir(eomt_folder)
sys.path.insert(0, repo_path)
sys.path.insert(0, eomt_folder)

eval_path = os.path.join(repo_path, "eval")
if eval_path not in sys.path:
    sys.path.append(eval_path)



 STEP 5: Fine-tuning COCO model on Cityscapes
Mounting Google Drive and setting up paths...
Mounted at /content/drive


# **HYPERPARAMETERS & STRATEGY**

In [ ]:

STRATEGY = "head_only"
IMG_SIZE = (640, 640)
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_WORKERS = 2
MAX_EPOCHS = 10 if STRATEGY == "head_only" else 15


CKPT_DIR = Path(project_folder) / "checkpoints" / f"step5_{STRATEGY}_lightning"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
last_ckpt_path = CKPT_DIR / "last.ckpt"
resume_path = str(last_ckpt_path) if last_ckpt_path.exists() else None


# **LOAD CONFIGURATIONS & DATASET**

In [ ]:

config_cs_path = os.path.join(eomt_folder, "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml")
with open(config_cs_path, "r") as f:
    config_cs = yaml.safe_load(f)

print("Loading Cityscapes datamodule")
data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})
data_module_kwargs["img_size"] = IMG_SIZE

data = data_module(
    path=data_path,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    check_empty_targets=False,
    **data_module_kwargs
)
data.setup()

Loading Cityscapes datamodule


# **BUILD MODEL & RAM LOGIC**

In [ ]:

from training.mask_classification_semantic import MaskClassificationSemantic

encoder_cfg = config_cs["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=IMG_SIZE, **encoder_cfg.get("init_args", {}))

network_cfg = config_cs["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=19,
    encoder=encoder,
    **network_kwargs,
)


if resume_path:
    print(f"\nRESUME FOUND: SKIP LOADING OF COCO WEIGHTS")
else:
    coco_path = os.path.join(project_folder, "eomt_coco.bin")
    print(f" Caricamento pesi COCO da: {coco_path}")
    coco_weights = torch.load(coco_path, map_location="cpu", weights_only=True)

    if "state_dict" in coco_weights:
        coco_weights = coco_weights["state_dict"]

    coco_weights = {k.replace("._orig_mod", ""): v for k, v in coco_weights.items()}
    skip_patterns = ["class_head", "empty_weight", "network.q", "class_predictor"]
    filtered_weights = {k: v for k, v in coco_weights.items() if not any(pattern in k for pattern in skip_patterns)}

    network.load_state_dict(filtered_weights, strict=False)
    print(f" Pesi COCO caricati.")

    del coco_weights
    del filtered_weights

# pulizia della RAM di sistema
gc.collect()

if STRATEGY == "head_only":
    lr, llrd, lr_mult = 1e-4, 1.0, 0.0
else:
    lr, llrd, lr_mult = 1e-4, 0.8, 1.0

model = MaskClassificationSemantic(
    network=network,
    img_size=IMG_SIZE,
    num_classes=19,
    attn_mask_annealing_enabled=False,
    lr=lr,
    llrd=llrd,
    lr_mult=lr_mult,
    warmup_steps=[100, 200],
    ckpt_path=None,
    delta_weights=False,
    load_ckpt_class_head=False
)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


RESUME FOUND: SKIP LOADING OF COCO WEIGHTS


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


# **LIGHTNING TRAINER**

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="best-miou-{epoch:02d}",
    save_top_k=1,
    save_last=True,
    monitor="metrics/val_iou_all",
    mode="max"
)

early_stop_callback = EarlyStopping(
    monitor="metrics/val_iou_all",
    min_delta=0.001,
    patience=3,
    verbose=True,
    mode="max"
)

import wandb
wandb_id_file = CKPT_DIR / "wandb_id.txt"
if wandb_id_file.exists():
    with open(wandb_id_file, "r") as f: run_id = f.read().strip()
    print(f"ID WandB exist: {run_id}")
else:
    run_id = wandb.util.generate_id()
    with open(wandb_id_file, "w") as f: f.write(run_id)

logger = WandbLogger(
    project="anomaly-segmentation-step5",
    name=f"lightning-{STRATEGY}",
    id=run_id,
    resume="allow"
)

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    accumulate_grad_batches=GRAD_ACCUM,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=logger,
    log_every_n_steps=10
)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


ID WandB exist: f2gyumbz


# **EXECUTE TRAINING WITH AUTO-RESUME**

In [ ]:
_original_torch_load = torch.load
def _hacked_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _hacked_torch_load
print("PyTorch Security Override")

if resume_path:
    print(f"\nRESUME FOUND: Resuming training from {last_ckpt_path.name}")
else:
    print("\nNo checkpoint found. Starting from scratch.")

#clear the last remaining bit of RAM before launching Lightning
gc.collect()

trainer.fit(
    model,
    data.train_dataloader(),
    data.val_dataloader(),
    ckpt_path=resume_path
)

torch.load = _original_torch_load



PyTorch Security Override

RESUME FOUND: Resuming training from last.ckpt


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 93.5 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 93.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 93.5 M                                                                                               
Total estimated model params size (MB): 373.995                                                                    
Modules in train mode: 301                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO: Restored all states from the checkpoint at /content/drive/MyDrive/FAIMDL Project/checkpoints/step5_head_only_lightning/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restored all states from the checkpoint at /content/drive/MyDrive/FAIMDL Project/checkpoints/step5_head_only_lightning/last.ckpt


Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


# **Final Evauation & Comparison**

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from pathlib import Path
from iouEval import iouEval
from torch.amp.autocast_mode import autocast

CITYSCAPES_CLASSES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle"
]
IGNORE_INDEX = 255
NUM_CLASSES = 19
IMG_SIZE = (640, 640)

def evaluate_model_weights_official(weights_path, is_lightning_ckpt=False, model_name="Model"):
    print(f"\n Evaluating {model_name}")
    print(f" Loading weights from: {Path(weights_path).name}")

    # Override di sicurezza
    _original_torch_load = torch.load
    torch.load = lambda *args, **kwargs: _original_torch_load(*args, **{**kwargs, 'weights_only': False})

    loaded_weights = torch.load(weights_path, map_location='cuda')

    if is_lightning_ckpt:
        state_dict = loaded_weights['state_dict']
        load_info = model.load_state_dict(state_dict)
        print(f"   -> [Check] Missing: {len(load_info.missing_keys)} | Unexpected: {len(load_info.unexpected_keys)}")
    else:
        if "state_dict" in loaded_weights:
            loaded_weights = loaded_weights["state_dict"]

        cleaned_state_dict = {}
        for k, v in loaded_weights.items():
            k = k.replace("._orig_mod", "")
            if k.startswith("network."):
                k = k.replace("network.", "", 1)

            #POSITIONAL EMBEDDING INTERPOLATION
            if "pos_embed" in k and v.shape != model.network.state_dict()[k].shape:
                print(f"   Risoluzione mismatch per {k}: {v.shape} -> {model.network.state_dict()[k].shape}")

                # B = Batch (1), N = Sequence Length (4096), C = Channels (768)
                B, N, C = v.shape

                # Calculating the side length of the original square
                hw_old = int(np.sqrt(N))

                # Calculating the side length of the target square from our model
                hw_new = int(np.sqrt(model.network.state_dict()[k].shape[1]))

                # Reshape a 2D grid: [1, 768, 64, 64]
                v_reshaped = v.permute(0, 2, 1).reshape(B, C, hw_old, hw_old)

                # Resize to smaller image: [1, 768, 40, 40]
                v_resized = F.interpolate(v_reshaped, size=(hw_new, hw_new), mode='bicubic', align_corners=False)

                # Flatten indietro a 1D sequence: [1, 1600, 768]
                v = v_resized.flatten(2).permute(0, 2, 1)

            cleaned_state_dict[k] = v

        load_info = model.network.load_state_dict(cleaned_state_dict, strict=False)
        print(f"   -> [Check] Missing: {len(load_info.missing_keys)} | Unexpected: {len(load_info.unexpected_keys)}")


    torch.load = _original_torch_load

    model.eval()
    model.cuda()

    #SETUP INFERENCE

    model.window_size = IMG_SIZE[0]
    evaluator = iouEval(20) # 19 classi + 1 (IGNORE_INDEX)

    with torch.no_grad():
        for batch in tqdm(data.val_dataloader(), desc=f"Inferencing {model_name}", leave=True):
            imgs = batch[0]
            targets = batch[1]
            target_list = model.to_per_pixel_targets_semantic(targets, IGNORE_INDEX)

            for b_idx in range(len(imgs)):
                img_tensor = imgs[b_idx].cuda()
                current_target = target_list[b_idx].cuda()

                #Sliding Window & Autocast

                with autocast(dtype=torch.float16, device_type="cuda"):
                    img_list = [img_tensor]
                    img_sizes = [img.shape[-2:] for img in img_list]

                    crops, origins = model.window_imgs_semantic(img_list)
                    mask_logits_per_layer, class_logits_per_layer = model(crops)
                    mask_logits = F.interpolate(mask_logits_per_layer[-1], IMG_SIZE, mode="bilinear")
                    crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
                    logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)

                    pred_logits = logits[0]

                # Resize
                if current_target.shape[-2:] != pred_logits.shape[-2:]:
                    pred_logits = F.interpolate(pred_logits.unsqueeze(0).float(), size=current_target.shape[-2:], mode='bilinear').squeeze(0)

                # Prepare dimension for evaluator [Batch=1, Channel=1, H, W]
                preds = pred_logits.argmax(0).unsqueeze(0).unsqueeze(0)
                target_mask = current_target.unsqueeze(0).unsqueeze(0)

                # Transforming 255 in 19 to avoid CUDA explodes
                target_mask[target_mask == IGNORE_INDEX] = 19

                evaluator.addBatch(preds, target_mask)

    mean_iou, iou_per_class = evaluator.getIoU()


    return iou_per_class[:19].cpu().numpy()

# **Execution Pipeline**

In [ ]:
# 1. Evaluating first the original model (requires interpolation)
original_ckpt = "/content/drive/MyDrive/FAIMDL Project/eomt_cityscapes.bin"
iou_original = evaluate_model_weights_official(original_ckpt, is_lightning_ckpt=False, model_name="Original Provided Model")

# 2. evaluating our model
finetuned_ckpt = finetuned_ckpt = "/content/drive/MyDrive/FAIMDL Project/checkpoints/step5_head_only_lightning/last.ckpt"
iou_finetuned = evaluate_model_weights_official(finetuned_ckpt, is_lightning_ckpt=True, model_name="Fine-Tuned Model")


# Print Comparative Table

print("\n" + "="*70)
print("FINAL COMPARISON: mIoU PER CLASS (CITYSCAPES)")
print("="*70)
print(f"{'Class Name'.ljust(15)} | {'Original Model'.rjust(22)} | {'Fine-Tuned Model'.rjust(22)}")
print("-" * 70)
for i, name in enumerate(CITYSCAPES_CLASSES):
    orig_val = f"{iou_original[i]*100:5.2f}%"
    ft_val = f"{iou_finetuned[i]*100:5.2f}%"
    print(f" {name.ljust(13)} | {orig_val.rjust(22)} | {ft_val.rjust(22)}")
print("-" * 70)
print(f"{'GLOBAL mIoU'.ljust(13)} | {np.nanmean(iou_original)*100:21.2f}% | {np.nanmean(iou_finetuned)*100:21.2f}%")
print("="*70)


 Evaluating Original Provided Model
 Loading weights from: eomt_cityscapes.bin
   Risoluzione mismatch per encoder.backbone.pos_embed: torch.Size([1, 4096, 768]) -> torch.Size([1, 1600, 768])
   -> [Check] Missing: 0 | Unexpected: 1


Inferencing Original Provided Model: 100%|██████████| 250/250 [04:14<00:00,  1.02s/it]



 Evaluating Fine-Tuned Model
 Loading weights from: last.ckpt
   -> [Check] Missing: 0 | Unexpected: 0


Inferencing Fine-Tuned Model: 100%|██████████| 250/250 [04:14<00:00,  1.02s/it]


FINAL COMPARISON: mIoU PER CLASS (CITYSCAPES)
Class Name      |         Original Model |       Fine-Tuned Model
----------------------------------------------------------------------
 road          |                 98.22% |                 98.04%
 sidewalk      |                 86.33% |                 84.65%
 building      |                 93.28% |                 92.09%
 wall          |                 66.93% |                 62.20%
 fence         |                 65.56% |                 60.57%
 pole          |                 65.75% |                 59.67%
 traffic light |                 70.57% |                 63.31%
 traffic sign  |                 77.51% |                 73.20%
 vegetation    |                 92.13% |                 91.42%
 terrain       |                 66.00% |                 66.26%
 sky           |                 94.81% |                 94.11%
 person        |                 81.93% |                 78.06%
 rider         |                 66.